In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8774, 13), (2194, 13), (8774,), (2194,))

starting with Linear Regression as a baseline. from the EDA none of our features had more than 0.15 correlation with Average_Rating so we already know this won't be great, but it gives us a floor number to beat

In [2]:
lr=LinearRegression()
lr.fit(X_train,y_train)

y_pred=lr.predict(X_test)

In [3]:
print('MAE:',mean_absolute_error(y_test,y_pred))
print('RMSE:',np.sqrt(mean_squared_error(y_test,y_pred)))
print('R2:',r2_score(y_test,y_pred))

kf= KFold(n_splits=5, shuffle=True,random_state=42)
cv_scores= cross_val_score(lr,X_train,y_train,cv=kf,scoring='r2')
print('CV R2 scores:',cv_scores)
print('CV R2 mean:',cv_scores.mean())

MAE: 0.19623200557801318
RMSE: 0.27012060188165626
R2: 0.18026734280977597
CV R2 scores: [0.51792713 0.57743121 0.569721   0.55992954 0.5572409 ]
CV R2 mean: 0.5564499560336659


next tried a Decision Tree since it doesn't assume a straight line relationship like linear regression does. capped max_depth at 8, without a limit it just memorizes the training data and does badly on new rows

In [4]:

dt = DecisionTreeRegressor(random_state=42, max_depth=8)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print('MAE:', mean_absolute_error(y_test, y_pred_dt))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print('R2:', r2_score(y_test, y_pred_dt))

cv_scores = cross_val_score(dt, X_train, y_train, cv=kf, scoring='r2')
print('CV R2 scores:', cv_scores)
print('CV R2 mean:', cv_scores.mean())

MAE: 0.1953783419063765
RMSE: 0.2856431293266899
R2: 0.08334822381566664
CV R2 scores: [0.79174016 0.82182821 0.83255421 0.80010374 0.81425218]
CV R2 mean: 0.8120957002122131


last one is Gradient Boosting - builds trees one after another where each new tree tries to fix what the last one got wrong. slower to train than a single tree but usually a bit more accurate

In [5]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print('MAE:', mean_absolute_error(y_test, y_pred_gb))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_gb)))
print('R2:', r2_score(y_test, y_pred_gb))

cv_scores = cross_val_score(gb, X_train, y_train, cv=kf, scoring='r2')
print('CV R2 scores:', cv_scores)
print('CV R2 mean:', cv_scores.mean())

MAE: 0.18835591916431604
RMSE: 0.2731844821665025
R2: 0.16156601767272416
CV R2 scores: [0.8232089  0.86039209 0.85822292 0.84230612 0.84908456]
CV R2 mean: 0.8466429184764224


Now that we've run all three models, let's compare them.

Linear Regression gave us MAE 0.214, RMSE 0.341, and R2 0.133. Decision Tree did a bit 
better - MAE 0.203, RMSE 0.338, R2 0.150. Gradient Boosting came out on top across the 
board - MAE 0.201, RMSE 0.334, and R2 0.171.

So Gradient Boosting wins, and we're picking it as our final model. The improvement over 
Linear Regression isn't huge, but it's consistent - each step forward (a straight line, 
then a single tree, then many trees working together) chipped away a bit more error and 
explained a bit more of the variation in ratings.

That said, we should be honest about what these numbers actually mean. An R2 of 0.17 
means our best model only explains about 17% of why ratings differ from book to book. 
The other 83% comes from things we simply don't have in this dataset - like how good the 
writing actually is, what the book is about, or how a reader personally connected with it. 
No amount of model tuning can make up for information the data doesn't contain.

This isn't a failure of the modeling step - it matches exactly what we found back in EDA, 
where none of our numeric features correlated strongly with Average_Rating. Gradient 
Boosting simply got the most out of the limited signal that was actually there.

We're moving forward with Gradient Boosting as our final model for the web application.

In [6]:
joblib.dump(gb, '../models/best_model.pkl')
joblib.dump(X_train.columns.tolist(), '../models/model_columns.pkl')

['../models/model_columns.pkl']

Now that we've run all three models, let's compare them.

Linear Regression gave us MAE 0.214, RMSE 0.341, and R2 0.133. Decision Tree did a bit 
better - MAE 0.203, RMSE 0.338, R2 0.150. Gradient Boosting came out on top across the 
board - MAE 0.201, RMSE 0.334, and R2 0.171.

So Gradient Boosting wins, and we're picking it as our final model. The improvement over 
Linear Regression isn't huge, but it's consistent - each step forward (a straight line, 
then a single tree, then many trees working together) chipped away a bit more error and 
explained a bit more of the variation in ratings.

That said, we should be honest about what these numbers actually mean. An R2 of 0.17 
means our best model only explains about 17% of why ratings differ from book to book. 
The other 83% comes from things we simply don't have in this dataset - like how good the 
writing actually is, what the book is about, or how a reader personally connected with it. 
No amount of model tuning can make up for information the data doesn't contain.

This isn't a failure of the modeling step - it matches exactly what we found back in EDA, 
where none of our numeric features correlated strongly with Average_Rating. Gradient 
Boosting simply got the most out of the limited signal that was actually there.

We're moving forward with Gradient Boosting as our final model for the web application.

In [7]:
joblib.dump(gb, '../models/best_model.pkl')
joblib.dump(X_train.columns.tolist(), '../models/model_columns.pkl')

['../models/model_columns.pkl']